In [1]:
"""
tools_mcp.py
============
CCA-F Project 4 - Design Tools and an MCP Server for a Claude Agent

This single file is the whole deliverable. It shows the four things the rubric
cares about, kept deliberately small so each idea is easy to read:

  1. TOOL DESIGN
     A tight set of 5 tools. The DESCRIPTION is the interface the model programs
     against - the agent never sees this code, only tool names + descriptions -
     so the descriptions are written to be specific, action-oriented, and
     non-overlapping. Invariants like idempotency are stated out loud, because
     the model will use whatever you tell it.

  2. tool_choice MODES
     - "auto"  : model decides whether/which tool to call   (the default)
     - "any"   : model MUST call some tool, but we don't say which
     - "tool"  : model MUST call one specific named tool     (e.g. structured output)

  3. MCP SERVER
     The exact same tool functions are exposed through a local MCP server
     (FastMCP) plus one resource. Secrets come from environment variables at the
     boundary - never hardcoded.

  4. STRUCTURED / RETRYABLE ERRORS
     Tools return a structured error carrying an `isRetryable` flag so the agent
     (via the dispatcher) knows when to retry a transient failure and when to
     give up on a permanent one.

Design note (why it's structured this way):
  There is ONE set of plain Python functions (the "core" tools). Two thin
  adapters sit on top of them:
     - the Claude loop  (Anthropic SDK)  -> calls them through a dispatch table
     - the MCP server   (FastMCP)        -> registers the same functions as tools
  That keeps the tool logic in one place and avoids over-architecting. No MCP
  client transport, no class hierarchy - just functions + two adapters.

How to run:
    # 1. Install deps
    pip install anthropic mcp

    # 2. Set secrets in the environment (never in code)
    export ANTHROPIC_API_KEY=sk-ant-...
    export BILLING_API_KEY=whatever-your-billing-service-uses
    export CLAUDE_MODEL=claude-sonnet-4-6      # optional; override per your access

    # 3a. Run the demos (auto / any / named tool_choice + retry behaviour)
    python tools_mcp.py

    # 3b. Start the MCP server instead (stdio transport)
    python tools_mcp.py serve
"""

import json
import os

# The Anthropic SDK is only needed for the live loop/demos, so import it lazily
# where it's used. FastMCP is imported in a try/except further down so the file
# still loads (and the demos still run) even if `mcp` isn't installed yet.


# The model is configuration, so it comes from the environment with a sane
# default. Point CLAUDE_MODEL at whatever model your account can call.
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-6")

# How many times the dispatcher retries a tool call that failed with a
# RETRYABLE error before giving up. Small on purpose.
MAX_RETRIES = 2


# ---------------------------------------------------------------------------
# 4. STRUCTURED ERRORS
# ---------------------------------------------------------------------------
# Every tool returns a plain dict. Success returns whatever data makes sense.
# Failure returns this exact shape so the caller can branch on `isRetryable`
# without parsing prose:
#
#     {"error": {"message": "...", "isRetryable": true|false}}
#
# isRetryable=True  -> transient (timeout, rate limit, blip). Retry may fix it.
# isRetryable=False -> permanent (bad id, missing config). Retrying is pointless.
def make_error(message, *, is_retryable):
    """Build the one canonical error shape used by every tool."""
    return {"error": {"message": message, "isRetryable": is_retryable}}


# ---------------------------------------------------------------------------
# Fake data store
# ---------------------------------------------------------------------------
# Stand-ins for a real billing backend. The point of the project is the tool
# LAYER, not the database, so this is intentionally trivial and in-memory.
_SUBSCRIPTIONS = {
    "cus_100": [
        {"id": "sub_a", "plan": "Pro Monthly",   "status": "active", "started": "2026-06-01"},
        {"id": "sub_b", "plan": "Extra Storage", "status": "active", "started": "2026-07-10"},
    ],
}
_CHARGES = {
    "ch_900": {"customer": "cus_100", "amount_cents": 4200, "refunded": False},
    # Used only to demonstrate the retry path (see issue_refund below).
    "ch_flaky": {"customer": "cus_100", "amount_cents": 999, "refunded": False},
}
_CONTACTS = {
    "cus_100": {"name": "Alex Rivera", "email": "alex@example.com"},
}

# Counts attempts against ch_flaky so we can make the FIRST attempt fail with a
# retryable error and the SECOND succeed - a deterministic transient failure.
_flaky_attempts = {}


# ---------------------------------------------------------------------------
# 1. THE CORE TOOLS  (5 of them - the sweet spot; more than ~5 and selection
#    accuracy drops as descriptions start to overlap)
# ---------------------------------------------------------------------------
# Each function's DOCSTRING is doing double duty: it's the description the MCP
# server hands to clients. The matching Anthropic JSON schema below repeats the
# same description for the Messages API. Keep the two in sync (a bigger system
# would generate one from the other; here we keep both explicit for clarity).
#
# Scopes are kept from overlapping on purpose: cancel does NOT refund, refund
# does NOT cancel, and each says so, so the model never has to guess.

def list_subscriptions(customer_id: str) -> dict:
    """List a customer's subscriptions, newest first. Read-only and idempotent:
    safe to call any time to see what a customer has before changing anything."""
    subs = _SUBSCRIPTIONS.get(customer_id)
    if subs is None:
        # Unknown id is a permanent error - retrying won't invent the customer.
        return make_error(f"No customer named {customer_id!r}.", is_retryable=False)
    newest_first = sorted(subs, key=lambda s: s["started"], reverse=True)
    return {"customer_id": customer_id, "subscriptions": newest_first}


def cancel_subscription(subscription_id: str) -> dict:
    """Cancel exactly one subscription by its subscription_id. Idempotent:
    cancelling an already-cancelled subscription is a no-op that still succeeds.
    Does NOT issue refunds."""
    for subs in _SUBSCRIPTIONS.values():
        for sub in subs:
            if sub["id"] == subscription_id:
                # Idempotent: if it's already cancelled we report success, not
                # an error, so the agent can safely retry without side effects.
                already = sub["status"] == "cancelled"
                sub["status"] = "cancelled"
                return {"subscription_id": subscription_id,
                        "status": "cancelled",
                        "was_already_cancelled": already}
    return make_error(f"No subscription named {subscription_id!r}.", is_retryable=False)


def issue_refund(charge_id: str) -> dict:
    """Issue a refund for a single charge by charge_id. NOT idempotent - calling
    it twice refunds twice - so only call after confirming the charge has not
    already been refunded. Does NOT cancel subscriptions."""
    charge = _CHARGES.get(charge_id)
    if charge is None:
        return make_error(f"No charge named {charge_id!r}.", is_retryable=False)

    # --- Deterministic transient failure, purely to demo the retry path. ---
    # First attempt against ch_flaky "times out" (retryable); second succeeds.
    if charge_id == "ch_flaky":
        _flaky_attempts[charge_id] = _flaky_attempts.get(charge_id, 0) + 1
        if _flaky_attempts[charge_id] < 2:
            return make_error("Billing service timed out.", is_retryable=True)

    # Because this tool is not idempotent, guard the double-refund case
    # explicitly and treat it as a permanent error.
    if charge["refunded"]:
        return make_error(f"Charge {charge_id!r} was already refunded.", is_retryable=False)

    charge["refunded"] = True
    return {"charge_id": charge_id, "refunded_cents": charge["amount_cents"]}


def get_billing_contact(customer_id: str) -> dict:
    """Look up the billing contact (name + email) for a customer_id from the
    billing service. Read-only. Use to reach the account owner, not to change
    billing."""
    # SECRETS FROM THE ENVIRONMENT, NOT THE CODE.
    # A real call to the billing service needs a key. We read it at call time
    # from the environment. If it's missing that's an operator/config problem,
    # so it's a permanent (non-retryable) error - retrying won't create a key.
    api_key = os.environ.get("BILLING_API_KEY")
    if not api_key:
        return make_error("BILLING_API_KEY is not set in the environment.",
                          is_retryable=False)

    contact = _CONTACTS.get(customer_id)
    if contact is None:
        return make_error(f"No contact for {customer_id!r}.", is_retryable=False)
    return {"customer_id": customer_id, **contact}


def record_resolution(category: str, action_taken: str, resolved: bool) -> dict:
    """Record the final structured resolution of this support case. Call once at
    the very end with the category, the action taken, and whether the issue is
    resolved."""
    # This tool exists to be FORCED with a named tool_choice. Forcing a specific
    # tool is the standard way to make the model emit structured output: instead
    # of hoping it writes clean JSON in prose, you require it to fill in this
    # tool's schema.
    return {"category": category, "action_taken": action_taken, "resolved": resolved}


# The dispatch table: tool name -> core function. One obvious place that maps a
# name the model chose to the code that runs. This is what the Claude loop uses.
TOOL_FUNCTIONS = {
    "list_subscriptions":  list_subscriptions,
    "cancel_subscription": cancel_subscription,
    "issue_refund":        issue_refund,
    "get_billing_contact": get_billing_contact,
    "record_resolution":   record_resolution,
}


# ---------------------------------------------------------------------------
# Anthropic tool schemas
# ---------------------------------------------------------------------------
# The Messages API needs an explicit JSON schema per tool. The `description`
# here is the single most important field for selection - it must match each
# function's docstring above.
TOOL_SCHEMAS = [
    {
        "name": "list_subscriptions",
        "description": list_subscriptions.__doc__,
        "input_schema": {
            "type": "object",
            "properties": {"customer_id": {"type": "string",
                                           "description": "The customer id, e.g. cus_100."}},
            "required": ["customer_id"],
        },
    },
    {
        "name": "cancel_subscription",
        "description": cancel_subscription.__doc__,
        "input_schema": {
            "type": "object",
            "properties": {"subscription_id": {"type": "string",
                                               "description": "The subscription id, e.g. sub_b."}},
            "required": ["subscription_id"],
        },
    },
    {
        "name": "issue_refund",
        "description": issue_refund.__doc__,
        "input_schema": {
            "type": "object",
            "properties": {"charge_id": {"type": "string",
                                         "description": "The charge id to refund, e.g. ch_900."}},
            "required": ["charge_id"],
        },
    },
    {
        "name": "get_billing_contact",
        "description": get_billing_contact.__doc__,
        "input_schema": {
            "type": "object",
            "properties": {"customer_id": {"type": "string",
                                           "description": "The customer id, e.g. cus_100."}},
            "required": ["customer_id"],
        },
    },
    {
        "name": "record_resolution",
        "description": record_resolution.__doc__,
        "input_schema": {
            "type": "object",
            "properties": {
                "category":     {"type": "string",
                                 "description": "Short category, e.g. 'cancellation' or 'refund'."},
                "action_taken": {"type": "string",
                                 "description": "One sentence on what was done."},
                "resolved":     {"type": "boolean",
                                 "description": "True if the customer's issue is fully resolved."},
            },
            "required": ["category", "action_taken", "resolved"],
        },
    },
]


# ---------------------------------------------------------------------------
# The dispatcher (where isRetryable is actually used)
# ---------------------------------------------------------------------------
def dispatch(name, args):
    """Run one tool by name, retrying transient (isRetryable) failures.

    Returns the tool's result dict (success or a final error). This is the one
    spot that reads the isRetryable flag and decides whether to try again.
    """
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return make_error(f"Unknown tool: {name}", is_retryable=False)

    for attempt in range(MAX_RETRIES + 1):
        result = fn(**args)
        is_error = isinstance(result, dict) and "error" in result
        if is_error and result["error"]["isRetryable"] and attempt < MAX_RETRIES:
            continue          # transient failure -> try again
        return result         # success, permanent error, or out of retries


# ---------------------------------------------------------------------------
# 2. THE CLAUDE LOOP  (Anthropic SDK integration)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = (
    "You are a billing support agent. Use the provided tools to inspect and "
    "change a customer's billing. Look things up before you change them. When "
    "the case is done, record the resolution."
)


def run_agent(user_message, *, tool_choice=None, max_turns=6, verbose=True):
    """Run a minimal agentic loop and return the final assistant message.

    `tool_choice` is applied on the FIRST turn only. That matters: if you leave
    "any" or a named "tool" forced on every turn, the model is compelled to keep
    calling tools forever and the loop can never reach a normal end_turn. So we
    force on turn one (to steer the first move) and relax to "auto" afterwards so
    the agent can finish.
    """
    import anthropic  # imported here so the file loads without the SDK present
    client = anthropic.Anthropic()

    if tool_choice is None:
        tool_choice = {"type": "auto"}

    messages = [{"role": "user", "content": user_message}]

    for _turn in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOL_SCHEMAS,
            tool_choice=tool_choice,
            messages=messages,
        )
        tool_choice = {"type": "auto"}  # relax after the first (possibly forced) turn

        messages.append({"role": "assistant", "content": response.content})

        # No tool requested -> the model gave its final answer. Done.
        if response.stop_reason != "tool_use":
            return response

        # Run every tool the model asked for and feed the results back.
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = dispatch(block.name, block.input)
                if verbose:
                    print(f"    tool: {block.name}({block.input}) -> {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                    # Tell the model this specific result was an error so it can
                    # react (e.g. surface it) rather than assume success.
                    "is_error": isinstance(result, dict) and "error" in result,
                })
        messages.append({"role": "user", "content": tool_results})

    return response  # hit max_turns


def _final_text(response):
    """Pull the plain-text blocks out of a response for tidy printing."""
    return "".join(b.text for b in response.content if b.type == "text").strip()


# ---------------------------------------------------------------------------
# 3. THE MCP SERVER  (same tools, exposed over MCP + one resource)
# ---------------------------------------------------------------------------
# We register the exact same core functions as MCP tools - their docstrings
# become the tool descriptions automatically. Wrapped in try/except so the rest
# of the file (and the demos) still work before `mcp` is installed.
try:
    from mcp.server.fastmcp import FastMCP

    mcp = FastMCP("billing-tools")

    # Register the identical core functions - no duplicated logic.
    for _fn in TOOL_FUNCTIONS.values():
        mcp.tool()(_fn)

    # One RESOURCE. Resources are read-only data (not actions). This one exposes
    # the refund policy so any MCP client can read it without a tool call.
    @mcp.resource("billing://policy")
    def refund_policy() -> str:
        """The refund policy an agent should follow."""
        return (
            "Refunds are allowed within 30 days of a charge. Cancelling a "
            "subscription does not automatically refund the last charge; issue a "
            "refund separately if the customer is owed one."
        )

except ImportError:
    mcp = None  # `pip install mcp` to enable `python tools_mcp.py serve`


# ---------------------------------------------------------------------------
# Demos
# ---------------------------------------------------------------------------
def demo_retry():
    """Show isRetryable in action WITHOUT needing the API.

    - ch_flaky fails once (retryable) then succeeds -> dispatcher recovers.
    - a bad charge id fails permanently -> dispatcher surfaces it immediately.
    """
    print("\n=== Structured / retryable errors (no API needed) ===")
    print("  Retryable transient failure (ch_flaky):")
    print("   ->", dispatch("issue_refund", {"charge_id": "ch_flaky"}))
    print("  Permanent failure (unknown charge):")
    print("   ->", dispatch("issue_refund", {"charge_id": "ch_nope"}))


def demo_auto():
    """tool_choice=auto: the model decides whether to use a tool at all.

    This is also the SELECTION demonstration - the descriptions alone should
    steer the model to list first, then cancel the newest subscription.
    """
    print("\n=== tool_choice = auto ===")
    print("  Task: cancel the customer's newest subscription (cus_100).")
    resp = run_agent(
        "Cancel the newest subscription for customer cus_100.",
        tool_choice={"type": "auto"},
    )
    print("  Final:", _final_text(resp))


def demo_any():
    """tool_choice=any: the model MUST call some tool (we don't say which).

    Useful when a step must result in an action, not a chat reply.
    """
    print("\n=== tool_choice = any ===")
    print("  Task: force at least one concrete tool call.")
    resp = run_agent(
        "Look into subscriptions for customer cus_100.",
        tool_choice={"type": "any"},
    )
    print("  Final:", _final_text(resp))


def demo_named():
    """tool_choice=tool: force ONE named tool - here, structured output."""
    print("\n=== tool_choice = named tool (structured output) ===")
    print("  Task: force record_resolution to get a structured result.")
    resp = run_agent(
        "The customer cus_100 asked to cancel Extra Storage; it's handled.",
        tool_choice={"type": "tool", "name": "record_resolution"},
    )
    print("  Final:", _final_text(resp) or "(structured tool call - see tool line above)")


def run_all_demos():
    demo_retry()  # always runs

    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("\n(Set ANTHROPIC_API_KEY to run the three tool_choice demos.)")
        return
    demo_auto()
    demo_any()
    demo_named()


# ---------------------------------------------------------------------------
# Entry point: run demos, or `serve` the MCP server (they can't share stdio).
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "serve":
        if mcp is None:
            print("MCP not installed. Run: pip install mcp")
            sys.exit(1)
        print("Starting MCP server 'billing-tools' on stdio...")
        mcp.run()  # stdio transport by default
    else:
        run_all_demos()


=== Structured / retryable errors (no API needed) ===
  Retryable transient failure (ch_flaky):
   -> {'charge_id': 'ch_flaky', 'refunded_cents': 999}
  Permanent failure (unknown charge):
   -> {'error': {'message': "No charge named 'ch_nope'.", 'isRetryable': False}}

=== tool_choice = auto ===
  Task: cancel the customer's newest subscription (cus_100).
    tool: list_subscriptions({'customer_id': 'cus_100'}) -> {'customer_id': 'cus_100', 'subscriptions': [{'id': 'sub_b', 'plan': 'Extra Storage', 'status': 'active', 'started': '2026-07-10'}, {'id': 'sub_a', 'plan': 'Pro Monthly', 'status': 'active', 'started': '2026-06-01'}]}
    tool: cancel_subscription({'subscription_id': 'sub_b'}) -> {'subscription_id': 'sub_b', 'status': 'cancelled', 'was_already_cancelled': False}
    tool: record_resolution({'category': 'cancellation', 'action_taken': 'Cancelled the newest subscription (sub_b, Extra Storage) for customer cus_100.', 'resolved': True}) -> {'category': 'cancellation', 'action_t